# PennyLane QNodes and analytic measurements

Bind one quantum function to default.qubit and MettleQ, then compare state, probabilities, expectation, and variance.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

A PennyLane QNode binds a quantum function to a device. Replacing default.qubit with MettleQ leaves the circuit function unchanged.

In [2]:
def make_qnode(device):
    @qml.qnode(device)
    def circuit(theta):
        qml.Hadamard(0)
        qml.CNOT(wires=[0, 1])
        qml.Rot(theta, -0.21, 0.13, wires=1)
        return qml.state(), qml.probs(wires=[1, 0]), qml.expval(qml.X(0) @ qml.Z(1)), qml.var(qml.Z(0))
    return circuit

reference_device = qml.device("default.qubit", wires=2)
reference_qnode = make_qnode(reference_device)

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(lambda: reference_qnode(0.31))

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: mettleq_qnode(0.31))
errors = [max_abs_error(left, right) for left, right in zip(reference, candidate)]
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

State, probabilities, expectation value, and variance are all compared.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/01_qnodes_and_measurements.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="all analytic measurements atol=2e-6",
    passed=max(errors) <= 2e-6,
    exact_match=all(np.array_equal(np.asarray(left), np.asarray(right)) for left, right in zip(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"per_measurement_max_errors": errors},
)


Comparison summary
------------------
Correctness contract: PASS — all analytic measurements atol=2e-6
SDK reference median: 0.753 ms
MettleQ median:       4.091 ms
Timing interpretation: the SDK reference was 5.436x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "all analytic measurements atol=2e-6", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"per_measurement_max_errors": [1.8000207054230655e-08, 2.718608671514744e-10, 1.181327383137365e-08, 1.2212453270876722e-15]}, "mettleq_median_ms": 4.091332986718044, "notebook": "pennylane/01_qnodes_and_measurements.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 0.7526669942308217, "reference_over_mettleq": 0.1839662028669514, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

This two-wire example teaches device substitution; it is too small to amortize MettleQ planning.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.